# Enterprise Content AI Prototype

This project demonstrates a **multi-agent AI system** designed to automate the complete lifecycle of enterprise content.

### Workflow Includes:
- Content drafting (AI generation)
- Brand & compliance review
- Human-in-the-loop approval
- Localization
- Multi-channel distribution
- Analytics & engagement insights

This notebook simulates a **real-world enterprise content pipeline** with reliability and fallback mechanisms.

In [ ]:
!pip install --upgrade google-generativeai

import os
import time
import random
import google.genai as genai
from google.colab import userdata

# Setup API
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## How to Use

1. Enter a **content prompt** (product update, blog, campaign, etc.)
2. System generates → reviews → asks for approval
3. Then performs localization, distribution & analytics
4. Final output is shown as a **content dashboard**

In [ ]:
# -----------------------
# Safe AI Call (with fallback)
# -----------------------
def safe_generate(prompt, retries=2, delay=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"Retry {attempt+1}... ({e})")
            time.sleep(delay)

    # Fallback (ensures demo never fails)
    return f"""
⚠️ AI temporarily unavailable — showing simulated output.

{prompt[:300]}...

[This fallback ensures uninterrupted workflow]
"""

# -----------------------
# Agents
# -----------------------

def draft_content(user_prompt):
    return safe_generate(f"Create professional enterprise content:\n{user_prompt}")

def review_content(draft):
    return safe_generate(f"""
Review this content for:
- Brand tone
- Terminology
- Compliance

Improve and return final version:

{draft}
""")

def human_approval(content):
    print("\n--- Review & Approval ---\n")
    print(content)
    decision = input("\nApprove? (yes/no): ").lower()

    if decision == "yes":
        return content
    else:
        return input("\nEnter revised version:\n")

def localize_content(content, lang="Spanish"):
    return safe_generate(f"Translate to {lang} (professional tone):\n{content}")

def distribute_content(content, channels):
    print("\n--- Distribution ---\n")
    for c in channels:
        print(f"✔ Published to {c}")

def analytics(content):
    return safe_generate(f"""
Analyze this content for engagement improvement:
- Tone
- Format
- Audience targeting

Content:
{content}
""")

def simulate_metrics():
    return {
        "Views": random.randint(1000, 5000),
        "Clicks": random.randint(100, 500),
        "Shares": random.randint(20, 150)
    }

## Run the Workflow

Enter your prompt below and experience the full AI pipeline.

In [ ]:
print("Enterprise Content AI Pipeline\n")

user_prompt = input("Enter content prompt: ")

channels = ["Website", "LinkedIn", "Twitter", "Email"]

# Pipeline
draft = draft_content(user_prompt)
reviewed = review_content(draft)
approved = human_approval(reviewed)
localized = localize_content(approved)
distribute_content(localized, channels)
feedback = analytics(localized)
metrics = simulate_metrics()

content_versions = {
    "Draft": draft,
    "Reviewed": reviewed,
    "Approved": approved,
    "Localized": localized,
    "Analytics": feedback
}

## Content Lifecycle Dashboard

A clean view of how content evolves through each stage, along with performance insights.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# -----------------------
# Content Stages (Clean Display)
# -----------------------
for stage, content in content_versions.items():
    display(Markdown(f"### {stage}"))
    display(Markdown(content))

# -----------------------
# Metrics Table
# -----------------------
display(Markdown("### 📊 Engagement Metrics"))

for k, v in metrics.items():
    display(Markdown(f"- **{k}:** {v}"))

# -----------------------
# Simple Chart (Clean)
# -----------------------
plt.figure(figsize=(6,4))
plt.bar(list(metrics.keys()), list(metrics.values()))
plt.title("Engagement Overview")
plt.ylabel("Count")

for i, v in enumerate(metrics.values()):
    plt.text(i, v, str(v), ha='center')

plt.show()